# 2. Denoising

In [ ]:
# Main imports
import numpy as np
import sys
from pathlib import Path
import datetime

# Scikit-learn imports
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, ShuffleSplit

# Add the project root to the Python path
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

# Import custom modules
from src.config import PROJECT_ROOT, DATA_ROOT, initialize_h5
from src.utilities import (
    read_datasets_from_label,
    get_keys_h5,
    repack_h5,
    create_denoise_config,
    save_denoise_results,
)
from src.processing import GaussianKernelSmoother

In [ ]:
# Directory and files setup

TYPE = 'tangle_00'
STAGE = '02_denoise'

INPUT_PATH = DATA_ROOT / TYPE / '01_preprocess.h5'
OUTPUT_PATH = DATA_ROOT / TYPE / (STAGE + '.h5')

print('INPUT_PATH: \t{:}'.format(INPUT_PATH))
print('OUTPUT_PATH: \t{:}'.format(OUTPUT_PATH))
print()

# Initialize output file
overwrite = True
verbose = True

initialize_h5(OUTPUT_PATH, overwrite=overwrite, verbose=verbose)

## Denoise Data

In [ ]:
# Denoising configuration

# Select datasets
selected_quality = ['good']

# Choose coordinate and passthrough keys
coordinate_keys = ['x', 'y', 'z']               # Denoise mass coordinates only
passthrough_keys = ['t', 'xp', 'yp', 'zp']     

# Method and cross-validation setup

# Gaussian kernel smoother
smoother_pipeline = Pipeline([
    ('smoother', GaussianKernelSmoother()),
])
param_grid = {'smoother__bandwidth': np.logspace(-3, -1, 25)}
cv_strategy = ShuffleSplit(n_splits=10, test_size=0.2, random_state=0)

smoother_cv = GridSearchCV(
    smoother_pipeline, 
    param_grid = param_grid, 
    cv = cv_strategy,
    scoring = 'neg_mean_squared_error',
    return_train_score = True,
    n_jobs = -1
)

# Saving configuration
hdf5_options={'compression':'gzip', 'compression_opts':9}

In [ ]:
# Denoise datasets

data_labels = get_keys_h5(INPUT_PATH)

for data_label in data_labels:
    
    # 1. Load data
    datasets, params = read_datasets_from_label(INPUT_PATH, data_label)
    if params['data_quality'] not in selected_quality:
        if verbose: 
            print('='*80)
            print(f"Skipping {data_label.ljust(10)} ({params['comments']})")
        continue

    t_series = datasets['t']
    if verbose: 
        print('='*80)
        print(f"{data_label.ljust(10)} ({params['comments']})")
        print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
        print('-'*80)

    # 2. Initialize output dictionaries
    denoised_datasets = {key: datasets[key] for key in passthrough_keys}    # Initialize with passthrough keys
    coordinate_cv_results = {key: {} for key in coordinate_keys}            # For storing cross-validation results

    # 3. Denoise each mass coordinate
    for coord_key in coordinate_keys:
        print(f"Processing coordinate: {coord_key}")

        # Fit and predict
        smoother_cv.fit(t_series.reshape(-1, 1), datasets[coord_key])
        denoised_datasets[coord_key] = smoother_cv.predict(t_series.reshape(-1, 1))
        
        # Extract best parameters and validation curves
        cv_results = {
            'best_params': smoother_cv.best_params_,
            'param_grid': np.array(smoother_cv.cv_results_[f'param_{list(smoother_cv.best_params_.keys())[0]}'], dtype=float),
            'mean_test_score': -smoother_cv.cv_results_['mean_test_score'],
            'std_test_score': smoother_cv.cv_results_['std_test_score'],
            'mean_train_score': -smoother_cv.cv_results_.get('mean_train_score', np.array([])),
            'std_train_score': smoother_cv.cv_results_.get('std_train_score', np.array([]))
        }
        print(f"  Best parameters: {cv_results['best_params']}")
        print(f"  Best test score: {cv_results['mean_test_score'].min()}")
        coordinate_cv_results[coord_key] = cv_results

    print(f"Processing {coordinate_keys} complete. Denoised datasets keys: {list(denoised_datasets.keys())}")
    print()

    # 4. Save denoised datasets and cv results
    denoise_config = create_denoise_config(coordinate_keys, passthrough_keys, smoother_pipeline, smoother_cv)
    save_denoise_results(OUTPUT_PATH, data_label, denoised_datasets, params, denoise_config, coordinate_cv_results, overwrite=overwrite, hdf5_options=hdf5_options)

repack_h5(OUTPUT_PATH, in_place=True)